In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

def is_checkout(path):
    return (path / 'train.py').is_file() and (path / 'src' / 'workrunner').is_dir()

cwd = Path.cwd()
ROOT = cwd if is_checkout(cwd) else next(path for path in sorted(cwd.iterdir()) if is_checkout(path))
sys.path.insert(0, str(ROOT / 'src'))
try:
    get_ipython().history_manager.enabled = False
    get_ipython().history_manager.db_log_output = False
except NameError:
    pass
print("Done importing")

Done importing


In [2]:
modules = {'numpy': 'numpy', 'torch': 'torch', 'pydantic': 'pydantic', 'loguru': 'loguru', 'cryptography': 'cryptography', 'ipykernel': 'ipykernel', 'Pillow': 'PIL', 'pyarrow': 'pyarrow'}
missing = [line for line in (ROOT / 'requirements.txt').read_text().splitlines() if importlib.util.find_spec(modules[line.split('>=')[0]]) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Ready')


Ready


In [3]:
# Check the V2Ray connection before training.
CUSTOM_V2RAY_LINK = ''  # Leave blank to test vpn.link from config.json.

def check_v2ray(root, custom_link):
    from http.client import HTTPException
    from time import perf_counter
    from urllib.error import HTTPError

    import workrunner
    if Path(workrunner.__file__).resolve().parent != (root / 'src/workrunner').resolve():
        print('Restart the kernel before using this source checkout.')
        return
    from workrunner.config import ConfigError
    from workrunner.login_config import load_training_config
    from workrunner.paths import set_checkout_directories
    from workrunner.socks_proxy import http_opener
    from workrunner.vpn import ManagedVPN, VPNError

    set_checkout_directories(root)
    try:
        vpn = load_training_config(root / 'config.json')['vpn']
    except ConfigError as error:
        print(f'VPN check could not load config.json. {error}')
        return
    if not vpn['check_url']:
        print('Set vpn.check_url in config.json to run the VPN check.')
        return

    link = custom_link.strip()
    source = 'custom link' if link else 'configured link'
    try:
        with ManagedVPN(vpn, link=link or None) as proxy:
            opener = http_opener(proxy)
            started = perf_counter()
            with opener.open(vpn['check_url'], timeout=10) as response:
                response.read(256)
                ping_ms = (perf_counter() - started) * 1000
    except VPNError as error:
        print(f'VPN check failed ({source}). {error}')
    except HTTPError as error:
        print(f'VPN check failed ({source}). Health endpoint returned HTTP {error.code}.')
    except (OSError, HTTPException) as error:
        print(f'VPN check failed ({source}). Health request failed ({type(error).__name__}).')
    else:
        print(f'VPN check passed ({source}). HTTPS ping to health endpoint {ping_ms:.0f} ms.')

check_v2ray(ROOT, CUSTOM_V2RAY_LINK)


VPN check passed (configured link). HTTPS ping to health endpoint 1383 ms.


In [4]:
import runpy

if 'trainer' in globals() and hasattr(trainer, 'stop_training'):
    trainer.stop_training()
    del trainer

# Set training.generate_artifacts to false in config.json for silent training.
workrunner = sys.modules.get('workrunner')
if workrunner is not None and Path(workrunner.__file__).resolve().parent != (ROOT / 'src/workrunner').resolve():
    raise RuntimeError('Restart the kernel before using this source checkout')
exit_code = runpy.run_path(str(ROOT / 'train.py'))['run'](ROOT)


1 | job-0 | N/A | 00:01:59 | 0
1 | job-0 | 0% | 00:01:58 | 0
1 | job-0 | 17% | 00:01:57 | 621102084346.21
1 | job-0 | 67% | 00:01:56 | 1536597226045.75
1 | job-0 | 67% | 00:01:55 | 2065350193692.59
1 | job-0 | 67% | 00:01:54 | 2350572982273.66
1 | job-0 | 68% | 00:01:53 | 2573939275978.98
1 | job-0 | 66% | 00:01:52 | 2708175491710.32
1 | job-0 | 68% | 00:01:51 | 2832190467792.43
1 | job-0 | 68% | 00:01:50 | 2909036975538.41
1 | job-0 | 66% | 00:01:49 | 2988211139213.29
1 | job-0 | 68% | 00:01:48 | 3037774280649.93
1 | job-0 | 67% | 00:01:47 | 3093690681868.41
1 | job-0 | 67% | 00:01:46 | 3127966217688.05
1 | job-0 | 68% | 00:01:45 | 3169453381491.93
1 | job-0 | 66% | 00:01:44 | 3194584317197.43
1 | job-0 | 68% | 00:01:43 | 3226544460298.38
1 | job-0 | 68% | 00:01:42 | 3244700516900.02
1 | job-0 | 66% | 00:01:41 | 3270329240571.77
1 | job-0 | 68% | 00:01:40 | 3284001576285.36
1 | job-0 | 68% | 00:01:39 | 3304402687485.17
1 | job-0 | 68% | 00:01:38 | 3298227519255.42
